# Terrain analysis: relative elevation model (REM)

Build a height-above-nearest-drainage (HAND-style) relative elevation surface and
summarize it by field (`rem_mean`) for subsequent stratification.

In [3]:
# Shared color palette
STREAM_BLUE = "#2171b5"
WET_CMAP = "Blues"
TERRAIN_CMAP = "terrain"
REM_CMAP = "RdYlBu_r"
NDWI_CMAP = "RdYlGn"
STRATA_COLORS = {
    "perennial": "#2166ac",
    "intermittent": "#74add1",
    "managed": "#4dac26",
    "non_partitioned": "#d9d9d9",
}
PARTITION_COLORS = {"pe": "#a6cee3", "et_gwsm": "#1f78b4", "et_irr": "#e31a1c"}

## 1. Relative elevation model (REM)

For a normal REM one creates a continuous, sloping water surface profile that follows a
specific river channel using Inverse Distance Weighting (IDW) — see
[RiverREM](https://github.com/OpenTopography/RiverREM) for an example.

Handily takes this further by combining NHD flowlines with NDWI-detected water pixels
to build a richer water-surface proxy across the valley:

```
REM = DEM - Z_w
```

where `Z_w` is a Gaussian-smoothed water-surface elevation derived from the stream mask
(NHD flowlines AND NDWI > threshold). Low REM → near-channel / shallow-GW access;
high REM → terrace / upland.

## 2. Setup

The DEM is fetched from a **local 3DEP STAC catalog** built once with:

```bash
handily stac build --states MT --stac-dir /nas/handily/stac/3dep_1m
```

`REMWorkflow` wraps three sequential steps that can also be called individually:

```python
workflow.fetch_vectors()   # clip NHD flowlines to AOI
workflow.fetch_dem()       # mosaic 3DEP tiles from local STAC
workflow.compute_rem()     # build stream mask + REM surface
```

In [4]:
import os
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from handily.config import HandilyConfig
from handily.io import aoi_from_bounds
from handily.pipeline import REMWorkflow

config_path = Path("beaverhead_config.toml")
with open(config_path, "rb") as f:
    cfg = tomllib.load(f)

config = HandilyConfig.from_dict(cfg)
bounds = tuple(cfg["bounds"])  # (W, S, E, N)

print(f"AOI bounds: {bounds}")
print(f"Output directory: {config.out_dir}")

AOI bounds: (-112.418, 45.445, -112.353, 45.49)
Output directory: /nas/handily/handily/beaverhead/outputs/


In [5]:
# Run the REM workflow
# cache_flowlines=True writes flowlines_bounds.fgb so subsequent runs skip the NHD clip.
# Set overwrite_outputs=True in the config to force re-computation.
aoi = aoi_from_bounds(bounds)
workflow = REMWorkflow(config, aoi)
results = workflow.run(
    ndwi_threshold=config.ndwi_threshold,
    flowlines_buffer_m=config.flowlines_buffer_m,
    cache_flowlines=True,
)

dem       = workflow.dem
flowlines = workflow.flowlines
ndwi      = workflow.ndwi
streams   = workflow.streams
rem       = workflow.rem
# workflow.stats = fields joined with zonal REM stats (has rem_mean column)
# workflow.fields = raw clipped fields (no rem_mean)
fields    = workflow.stats

print('Workflow complete')
print(f'  DEM shape:    {dem.shape}')
print(f'  Flowlines:    {len(flowlines)} segments')
print(f'  Fields:       {len(fields)} polygons')
print(f'  Columns:      {list(fields.columns)}')
print(f'  REM range:    {float(rem.min()):.1f} – {float(rem.max()):.1f} m')
print(f'  rem_mean:     {fields["rem_mean"].min():.2f} – {fields["rem_mean"].max():.2f} m')


/home/dgketchum/code/handily/.venv/lib/python3.11/site-packages/rasterstats/io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Workflow complete
  DEM shape:    (5798, 5864)
  Flowlines:    253 segments
  Fields:       48 polygons
  Columns:      ['FID', 'SOURCECODE', 'COUNTY_NO', 'COUNTYNAME', 'ITYPE', 'USAGE', 'MAPPEDBY', 'geometry', 'rem_mean']
  REM range:    0.0 – 14.0 m
  rem_mean:     0.64 – 11.52 m


In [6]:
# Save NB02 outputs — required by NB03/NB04
os.makedirs(out_dir, exist_ok=True)

rem.rio.to_raster(os.path.join(out_dir, 'rem_bounds.tif'), overwrite=True)
ndwi.rio.to_raster(os.path.join(out_dir, 'ndwi_bounds.tif'), overwrite=True)
fields.to_file(os.path.join(out_dir, 'fields_bounds.fgb'), driver='FlatGeobuf')
# Note: flowlines_bounds.fgb already written by pipeline (cache_flowlines=True)

print('Saved to', out_dir)
print('  rem_bounds.tif')
print('  ndwi_bounds.tif')
print('  fields_bounds.fgb')
print('  flowlines_bounds.fgb  (written by workflow.run)')


NameError: name 'out_dir' is not defined

## 3. DEM

USGS 3DEP 1 m LiDAR DEM. The entire valley slopes north, which is why we need the REM
rather than raw elevation to identify near-stream terrain.

In [ ]:
# DEM + hillshade
dem_arr = dem.values.astype(float)
dzdx = np.gradient(dem_arr, axis=1)
dzdy = np.gradient(dem_arr, axis=0)
slope = np.sqrt(dzdx**2 + dzdy**2)
aspect = np.arctan2(-dzdy, dzdx)
az, alt = np.radians(315), np.radians(45)
hillshade = np.sin(alt)*np.cos(slope) + np.cos(alt)*np.sin(slope)*np.cos(az - aspect)

# Derive spatial extent so hillshade aligns with the DEM coordinate frame
x_vals = dem.x.values; y_vals = dem.y.values
dem_extent = [x_vals[0], x_vals[-1], y_vals[-1], y_vals[0]]  # [left, right, bottom, top]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(hillshade, cmap='gray', extent=dem_extent, origin='upper', aspect='auto')
axes[0].set_title('Hillshade')
axes[0].set_xlabel(dem.x.attrs.get('long_name', 'x'))
axes[0].set_ylabel(dem.y.attrs.get('long_name', 'y'))
dem.plot(ax=axes[1], cmap=TERRAIN_CMAP, robust=True)
axes[1].set_title('DEM (m elevation)')
print(f'Elevation range: {float(dem.min()):.1f} – {float(dem.max()):.1f} m')
print(f'Hillshade shape: {hillshade.shape}')
plt.tight_layout(); plt.show()

## 4. NHD flowlines

Flowlines provide channel geometry and stream type via the FCODE attribute.

In [ ]:
from handily.nhd import FCODE_CATEGORIES

# FCODE distribution
fcode_col = "fcode" if "fcode" in flowlines.columns else "FCODE"
cats = flowlines[fcode_col].map(FCODE_CATEGORIES).fillna("unknown")
print("Flowline categories:")
print(cats.value_counts().to_string())

fig, ax = plt.subplots(figsize=(10, 7))
for cat, color in [("perennial", STRATA_COLORS["perennial"]),
                   ("intermittent", STRATA_COLORS["intermittent"]),
                   ("managed", STRATA_COLORS["managed"])]:
    sub = flowlines[cats == cat]
    if len(sub):
        sub.plot(ax=ax, color=color, linewidth=1.0, label=f"{cat} (n={len(sub)})")
ax.set_title("NHD Flowlines by Category")
ax.legend()
plt.tight_layout(); plt.show()

## 5. NDWI

NDWI constrains the wetted-channel mask. The threshold trades omission (missing real water)
against commission (wet vegetation, shadow). The Beaverhead example uses `0.45`.

In [ ]:
# NDWI + threshold sensitivity sweep
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ndwi.plot(ax=axes[0], cmap=NDWI_CMAP, robust=True)
axes[0].set_title("NDWI")

# Overlay contour lines for each threshold on top of the NDWI image.
# xarray's plot.contour() is coordinate-aware, so extents match exactly.
ndwi.plot(ax=axes[1], cmap=NDWI_CMAP, robust=True, add_colorbar=False)
threshold_colors = ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]
for t, c in zip([0.10, 0.20, 0.30, 0.45], threshold_colors):
    ndwi.plot.contour(ax=axes[1], levels=[t], colors=[c], linewidths=1.2)
    pct = float((ndwi > t).mean()) * 100
    axes[1].plot([], [], color=c, linewidth=1.5, label=f"NDWI > {t} ({pct:.1f}%)")
axes[1].set_title("NDWI Threshold Sensitivity")
axes[1].legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# NHD flowlines overlaid on NDWI
fig, ax = plt.subplots(figsize=(10, 7))
ndwi.plot(ax=ax, cmap=NDWI_CMAP, robust=True, add_colorbar=True)
flowlines.to_crs(ndwi.rio.crs).plot(ax=ax, color=STREAM_BLUE, linewidth=0.8)
ax.plot([], [], color=STREAM_BLUE, label="NHD flowlines")
ax.set_title("NHD Flowlines over NDWI")
ax.legend()
plt.tight_layout(); plt.show()

## 6. Stream mask

The stream mask is the AND of rasterized NHD flowlines and NDWI > threshold.
A pixel is "stream" only if it lies on an NHD segment **and** NDWI detects water there.
This removes dry channel sections that NHD maps but no water is present.

### NHD buffering

NHD flowlines can be misaligned with actual channel positions. Buffering before
rasterization captures nearby NDWI water pixels that would otherwise be missed.

In [ ]:
from rasterio import features

fl_proj = flowlines.to_crs(ndwi.rio.crs)
transform = ndwi.rio.transform()
water = (ndwi > config.ndwi_threshold).values.astype('uint8')

# Unbuffered
shapes_raw = [(g, 1) for g in fl_proj.geometry if g is not None]
nhd_raw = features.rasterize(shapes_raw, out_shape=ndwi.shape,
                              transform=transform, fill=0,
                              all_touched=True, dtype='uint8')

# Buffered
buf_m = config.flowlines_buffer_m or 10.0
fl_buf = fl_proj.copy(); fl_buf['geometry'] = fl_buf.geometry.buffer(buf_m)
shapes_buf = [(g, 1) for g in fl_buf.geometry if g is not None]
nhd_buf = features.rasterize(shapes_buf, out_shape=ndwi.shape,
                              transform=transform, fill=0,
                              all_touched=True, dtype='uint8')

and_raw = nhd_raw & water
and_buf = nhd_buf & water
gained  = (and_buf == 1) & (and_raw == 0)

# Derive spatial extent so all three panels share the NDWI coordinate frame
x_vals = ndwi.x.values; y_vals = ndwi.y.values
ndwi_extent = [x_vals[0], x_vals[-1], y_vals[-1], y_vals[0]]  # [left, right, bottom, top]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(and_raw, cmap='Greens', extent=ndwi_extent, origin='upper', aspect='auto')
axes[0].set_title(f'Unbuffered AND ({and_raw.sum():,} px)')
axes[1].imshow(and_buf, cmap='Greens', extent=ndwi_extent, origin='upper', aspect='auto')
axes[1].set_title(f'{buf_m:.0f} m buffer AND ({and_buf.sum():,} px)')
rgb = np.zeros((*and_raw.shape, 3), dtype='uint8')
rgb[and_raw == 1] = [0, 180, 0]
rgb[gained]       = [0, 220, 255]
axes[2].imshow(rgb, extent=ndwi_extent, origin='upper', aspect='auto')
axes[2].set_title(f'Green=both  Cyan=gained with buffer ({gained.sum():,} px)')
for ax in axes:
    ax.set_xlabel(ndwi.x.attrs.get('long_name', 'x'))
    ax.set_ylabel(ndwi.y.attrs.get('long_name', 'y'))
plt.tight_layout(); plt.show()

# Final stream mask
fig, ax = plt.subplots(figsize=(10, 7))
streams.plot(ax=ax, cmap=WET_CMAP)
ax.set_title(f'Stream Mask — NHD AND NDWI>{config.ndwi_threshold}  ({int(streams.sum()):,} stream pixels)')
plt.tight_layout(); plt.show()

## 7. Relative elevation model

1. Sample DEM elevations along stream-mask pixels.
2. Gaussian-smooth to a continuous water-surface proxy `Z_w`.
3. `REM = DEM − Z_w`.

| REM (m) | Landscape position | Shallow-GW influence |
|---|---|---|
| 0 – 1 | Near water surface | High |
| 1 – 2 | Low floodplain | Moderate – high |
| 2 – 5 | Intermediate terrace | Moderate |
| 5 – 10 | High terrace | Low |
| > 10 | Upland | Very low |

In [ ]:
# REM map + distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rem.plot(ax=axes[0], cmap=REM_CMAP, vmin=0, vmax=10)
axes[0].set_title("Relative Elevation Model (m)")

vals = rem.values.flatten()
vals = vals[~np.isnan(vals) & (vals > 0) & (vals < 20)]
axes[1].hist(vals, bins=80, color=STREAM_BLUE, edgecolor="none", alpha=0.7)
for q, label in [(0.25, "Q1"), (0.50, "Median"), (0.75, "Q3")]:
    v = float(np.quantile(vals, q))
    axes[1].axvline(v, color="red", linestyle="--", label=f"{label}: {v:.2f} m")
axes[1].axvline(config.rem_threshold, color="orange", linewidth=2,
                label=f"Partition threshold: {config.rem_threshold} m")
axes[1].set_xlabel("REM (m)"); axes[1].set_ylabel("Pixel count")
axes[1].set_title("REM Distribution")
axes[1].legend()
plt.tight_layout(); plt.show()

## 8. Field statistics

`rem_mean` (zonal mean of REM per field polygon) is the sampling and stratification input in NB03/NB04.
Fields with `rem_mean < rem_threshold` are designated as *partitioned* (shallow-GW influence).

In [ ]:
print(f"Fields: {len(fields)}")
print(f"Columns: {list(fields.columns)}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Choropleth
fields.plot(column="rem_mean", ax=axes[0], cmap=REM_CMAP, vmin=0, vmax=10,
            legend=True, legend_kwds={"label": "Mean REM (m)", "shrink": 0.6},
            edgecolor="black", linewidth=0.3)
axes[0].set_title("Fields — Mean REM")

# Histogram with partition threshold
fields["rem_mean"].hist(ax=axes[1], bins=30, color=STREAM_BLUE, edgecolor="white", alpha=0.8)
axes[1].axvline(config.rem_threshold, color="orange", linewidth=2,
                label=f"Threshold {config.rem_threshold} m")
n_part = (fields["rem_mean"] < config.rem_threshold).sum()
axes[1].set_xlabel("rem_mean (m)"); axes[1].set_ylabel("Field count")
axes[1].set_title(f"rem_mean distribution  ({n_part}/{len(fields)} partitioned)")
axes[1].legend()
plt.tight_layout(); plt.show()

### Operational callouts

**`handily bounds`** — runs the full pipeline from the CLI without writing Python:

```bash
handily bounds --bounds -112.42 45.44 -112.35 45.50 \
  --fields fields.gpkg --ndwi-dir /data/ndwi \
  --flowlines-local-dir /data/nhd --stac-dir /data/stac \
  --out-dir /tmp/bvr
```

**`handily aoi`** — tiles a field layer into AOI polygons for multi-watershed runs:

```bash
handily aoi --fields fields.gpkg --out aoi_tiles.fgb
```

> **Repo Capability — Notebook 2 (Terrain Analysis)**
> Modules exercised: `handily.pipeline`, `handily.compute`, `handily.stac_3dep`, `handily.dem`, `handily.nhd`
> Key functions: `REMWorkflow.run()`, `REMWorkflow.fetch_dem()`, `REMWorkflow.compute_rem()`,
> `build_streams_mask_from_nhd_ndwi()`, `compute_rem_quick()`
> CLI equivalents: `handily bounds`, `handily stac build/extend`, `handily aoi`

> **Artifacts Produced**
> - `{out_dir}/dem_bounds_1m.tif` — 1 m LiDAR DEM clipped to AOI
> - `{out_dir}/flowlines_bounds.fgb` — NHD flowlines clipped to AOI
> - `{out_dir}/streams_bounds.tif` — Combined NHD + NDWI stream mask
> - `{out_dir}/ndwi_bounds.tif` — NDWI mosaic clipped to AOI
> - `{out_dir}/rem_bounds.tif` — Relative Elevation Model
> - `{out_dir}/fields_bounds.fgb` — Field polygons with `rem_mean` column

## Key takeaways

- REM is a practical proxy for shallow-GW access at field scale.
- Constraining NHD flowlines with NDWI improves the wetted-channel mask.
- `rem_mean` provides a simple stratification metric; thresholds are study-specific.

**Next**: [Notebook 03 — Points Samplling](03_points_samplling.ipynb)